In [13]:
from manim import *
import numpy as np

class DynamicWaveAnimation(Scene):
    def construct(self):
        # १. शीर्षक (Title)
        title = Text(
            "तरंगको गहिरो विश्लेषण: y = A sin(ωt + φ)", 
            font="Noto Sans Devanagari", 
            font_size=38,
            color=BLUE
        ).to_edge(UP, buff=0.3)
        self.add(title)

        # २. बायाँतर्फ विस्तृत व्याख्या (Explanation)
        explanation = VGroup(
            Text("तरंगका मुख्य तत्वहरू:", font="Noto Sans Devanagari", font_size=26, color=YELLOW),
            VGroup(MathTex("A", font_size=40), Text(": आयाम (अधिकतम विस्थापन)", font="Noto Sans Devanagari", font_size=20)).arrange(RIGHT, buff=0.2),
            VGroup(MathTex("\\omega", font_size=40), Text(": कोणीय गति (घुम्ने दर)", font="Noto Sans Devanagari", font_size=20)).arrange(RIGHT, buff=0.2),
            VGroup(MathTex("\\phi", font_size=40), Text(": फेज (सुरुवाती अवस्था)", font="Noto Sans Devanagari", font_size=20)).arrange(RIGHT, buff=0.2),
            VGroup(MathTex("t", font_size=40), Text(": समयको प्रवाह", font="Noto Sans Devanagari", font_size=20)).arrange(RIGHT, buff=0.2),
        ).arrange(DOWN, aligned_edge=LEFT, buff=0.35).to_edge(LEFT, buff=0.7).shift(UP * 0.5)
        
        self.add(explanation)

        # ३. गतिशील मानहरू (Live Value Display - HGroup Error Fixed)
        A_var = ValueTracker(1.2)
        w_var = ValueTracker(1.5)
        p_var = ValueTracker(0)
        t_var = ValueTracker(0)

        # यहाँ HGroup को सट्टा VGroup(...).arrange(RIGHT) प्रयोग गरिएको छ
        value_display = VGroup(
            Text("प्रत्यक्ष तथ्याङ्क:", font="Noto Sans Devanagari", font_size=24, color=GOLD),
            VGroup(Text("A =", font_size=22), DecimalNumber(1.2, font_size=22).add_updater(lambda d: d.set_value(A_var.get_value()))).arrange(RIGHT, buff=0.2),
            VGroup(Text("ω =", font_size=22), DecimalNumber(1.5, font_size=22).add_updater(lambda d: d.set_value(w_var.get_value()))).arrange(RIGHT, buff=0.2),
            VGroup(Text("φ =", font_size=22), DecimalNumber(0.0, font_size=22).add_updater(lambda d: d.set_value(p_var.get_value()))).arrange(RIGHT, buff=0.2)
        ).arrange(DOWN, aligned_edge=LEFT, buff=0.25).next_to(explanation, DOWN, buff=0.8, aligned_edge=LEFT)
        
        self.add(value_display)

        # ४. एनिमेसन सेटिङ (दायाँतर्फ सारिएको)
        graph_origin = np.array([2.0, -1.0, 0])
        circle_center = graph_origin + LEFT * 3.0
        
        circle = always_redraw(lambda: Circle(radius=A_var.get_value(), color=BLUE_B, stroke_width=2).move_to(circle_center))
        x_axis = Line(start=circle_center, end=graph_origin + RIGHT * 4.5, color=GRAY_A)
        self.add(circle, x_axis)

        # ५. गतिशील बिन्दु र तरंग (Direction & Sync Fixed)
        # Dot on circle
        dot_circle = always_redraw(lambda: Dot(radius=0.08, color=RED).move_to(
            circle_center + A_var.get_value() * np.array([
                np.cos(w_var.get_value() * t_var.get_value() + p_var.get_value()), 
                np.sin(w_var.get_value() * t_var.get_value() + p_var.get_value()), 
                0
            ])
        ))

        # Synchronized Wave: y = A sin(wt - kx + p)
        # x_val=0 हुँदा sin(wt + p) हुन्छ, जसले वृत्तको बिन्दुसँग उचाइ मिलाउँछ
        wave_path = always_redraw(lambda: VMobject().set_color(YELLOW).set_points_smoothly(
            [np.array([
                graph_origin[0] + x_val, 
                graph_origin[1] + A_var.get_value() * np.sin(w_var.get_value() * t_var.get_value() - 2.5 * x_val + p_var.get_value()), 
                0
            ]) for x_val in np.linspace(0, 4.5, 100)]
        ))

        # Horizontal Sync Line
        horiz_line = always_redraw(lambda: DashedLine(
            start=dot_circle.get_center(),
            end=np.array([graph_origin[0], dot_circle.get_center()[1], 0]),
            color=WHITE, stroke_opacity=0.5
        ))

        self.add(dot_circle, wave_path, horiz_line)

        # ६. उप-शीर्षकहरू (Subtitles)
        subtitle = Text("एनिमेसन सुरु हुँदैछ...", font="Noto Sans Devanagari", font_size=24, color=GREEN).to_edge(DOWN, buff=0.5)
        self.add(subtitle)

        # --- एनिमेसन रन ---
        self.play(t_var.animate.set_value(5), run_time=5, rate_func=linear)
        
        # आयाम परिवर्तन
        self.play(Transform(subtitle, Text("आयाम (A) बढ्दा तरंगको उचाइ र ऊर्जा बढ्छ।", font="Noto Sans Devanagari", font_size=24, color=GREEN).to_edge(DOWN)))
        self.play(A_var.animate.set_value(2.0), run_time=3)
        self.play(A_var.animate.set_value(0.6), run_time=3)
        self.play(A_var.animate.set_value(1.2), run_time=1)

        # आवृत्ति परिवर्तन
        self.play(Transform(subtitle, Text("आवृत्ति (ω) बढ्दा तरंगहरू खाँदिएर आउँछन्।", font="Noto Sans Devanagari", font_size=24, color=GREEN).to_edge(DOWN)))
        self.play(w_var.animate.set_value(5.0), run_time=4)
        self.play(w_var.animate.set_value(1.5), run_time=2)

        # फेज परिवर्तन
        self.play(Transform(subtitle, Text("फेज (φ) ले तरंगलाई समय अक्षमा अगाडि वा पछाडि धकेल्छ।", font="Noto Sans Devanagari", font_size=24, color=GREEN).to_edge(DOWN)))
        self.play(p_var.animate.set_value(PI), run_time=3)
        self.play(p_var.animate.set_value(0), run_time=3)

        self.play(t_var.animate.set_value(10), run_time=5, rate_func=linear)
        self.wait(2)

%manim -qk -v warning DynamicWaveAnimation

Manim Community v0.19.1